# PawCity AI — Resumable Model Comparisons

**IEOR 4570 · Project 2 · Deep Research path**

This notebook runs the three required same-model comparisons while respecting a conservative API request pace. It automatically saves results after each baseline and each completed agent answer. If an API rate limit or network interruption occurs, rerun the comparison cell to resume from the latest checkpoint.

**Important:** The notebook contains no secret keys. Do not save or share an API key in project files.


## 1. Locate the project directory

Open this notebook from the extracted project directory containing `app.py`, `pawcity/`, and `data/`.


In [ ]:
from pathlib import Path
import os
import sys
import json
import runpy

ROOT = Path.cwd()
if not (ROOT / "app.py").exists() and (ROOT / "PawCity_AI_Gemini_VertexAI" / "app.py").exists():
    ROOT = ROOT / "PawCity_AI_Gemini_VertexAI"
if not (ROOT / "app.py").exists():
    raise FileNotFoundError("Open this notebook inside the extracted PawCity AI project folder.")
sys.path.insert(0, str(ROOT))
print("Project directory:", ROOT.resolve())


## 2. Configure Google Cloud Vertex AI

**No OpenAI API key or Gemini Developer API key is used.** Run `gcloud auth application-default login` in Mac Terminal, then set your real school GCP project ID below. Eligible school GCP credits may cover Vertex AI usage. The model is `gemini-3.5-flash-lite` by default.

If the previous comparison checkpoint was made with GPT, archive it before continuing. The migration helper preserves the GPT results outside the active Gemini checkpoint.


In [ ]:
from pathlib import Path
import json
import os

# Replace this placeholder with your actual Google Cloud school project ID.
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "")
if not PROJECT_ID:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT to your course Google Cloud Project ID before running this cell.")

os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["PAWCITY_MODEL"] = "gemini-3.5-flash-lite"
os.environ.setdefault("PAWCITY_API_RPM", "7")

checkpoint = ROOT / "reports" / "sample_comparisons.json"
if checkpoint.exists():
    saved = json.loads(checkpoint.read_text(encoding="utf-8"))
    if saved.get("model") != os.environ["PAWCITY_MODEL"]:
        raise RuntimeError(
            "This checkpoint belongs to another model. Move the old GPT file out of reports/ first."
        )
print("GCP project configured:", PROJECT_ID)
print("Vertex AI model:", os.environ["PAWCITY_MODEL"])


## 3. Run or resume all three comparisons

**Real, billable model calls happen here.** Progress is saved to `reports/sample_comparisons.json`. On interruption, rerun this cell. Finished comparisons are skipped, and saved bare-model baseline responses are reused.

This notebook does not invent or recover earlier responses that were never saved by an older version of the script.


In [ ]:
assert os.getenv("GOOGLE_CLOUD_PROJECT"), "The school GCP project is not configured."
runpy.run_path(str(ROOT / "scripts" / "compare_samples.py"), run_name="__main__")


## 4. Review all genuine model outputs

When all three comparisons complete, this step shows a concise summary. The unedited full outputs are saved in `reports/sample_comparisons.json` and the README comparison table. Check the wording, citations and dates before submission.


In [ ]:
from IPython.display import display

report_path = ROOT / "reports" / "sample_comparisons.json"
if not report_path.exists():
    print("No comparison checkpoint has been created yet.")
else:
    data = json.loads(report_path.read_text(encoding="utf-8"))
    rows = [{
        "Question ID": r["id"],
        "Baseline saved": bool(r.get("same_model_no_system_no_tools_answer")),
        "Agent answer saved": bool(r.get("agent_answer")),
        "Tool calls": len(r.get("agent_tool_calls", [])),
        "Citations": len(r.get("agent_citations", [])),
    } for r in data.get("comparisons", [])]
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except ImportError:
        for row in rows:
            print(row)
    print("Completed:", sum(r["Agent answer saved"] for r in rows), "/ 3")
